# Implement Agent Loops: Practice Exercise

In this exercise, you will implement the **agent loop** - the core mechanism that allows a ReAct agent to continue reasoning until it reaches a final answer.

**What you'll implement:**
- The `query` function that runs the Thought -> Action -> Observation loop
- Loop until the agent outputs an Answer (no more actions)
- Handle multiple tool calls in sequence

**Estimated time:** 10-15 minutes

## Setup

Run this cell to import all required libraries and configure the environment.

In [1]:
# Setup - run this cell first

from openai import OpenAI
import os
import re

IN_COLAB = 'COLAB_GPU' in os.environ or 'COLAB_TPU_ADDR' in os.environ


if not IN_COLAB:
  from dotenv import load_dotenv
  load_dotenv()
  # Verify OpenAI API key is set
  if not os.getenv("OPENAI_API_KEY"):
    ("WARNING: OPENAI_API_KEY environment variable not set!")
  else:
    OPEN_AI_API_KEY=os.getenv("OPENAI_API_KEY")
    print("OpenAI API key found. Ready to proceed!")
else:
  from google.colab import userdata
  OPEN_AI_API_KEY=userdata.get('OPEN_AI_API_KEY')

In [2]:
# Setup - run this cell

client = OpenAI(api_key=OPEN_AI_API_KEY)
print("Setup complete!")

Setup complete!


## Provided Code: Agent Class

The `MovieAdvisorAgent` class from the previous exercise is provided for you. This handles conversation management and API calls.

In [3]:
class MovieAdvisorAgent:
    """
    A ReAct agent that uses OpenAI to reason about movie recommendations.
    """

    def __init__(self, system_prompt: str = ""):
        self.system_prompt = system_prompt
        self.messages = []
        if self.system_prompt:
            self.messages.append({"role": "system", "content": system_prompt})

    def __call__(self, message: str) -> str:
        self.messages.append({"role": "user", "content": message})
        result = self.execute()
        self.messages.append({"role": "assistant", "content": result})
        return result

    def execute(self) -> str:
        completion = client.chat.completions.create(
            model="gpt-4o-mini",
            temperature=0,
            messages=self.messages
        )
        return completion.choices[0].message.content

print("Agent class loaded!")

Agent class loaded!


## Provided Code: Tools and Prompt

The tools and system prompt are provided. Note the system prompt now includes a multi-step example.

In [4]:
# Movie database with runtime information
movie_database = {
    "action": {"title": "Mad Max: Fury Road", "director": "George Miller", "runtime": 120},
    "comedy": {"title": "The Grand Budapest Hotel", "director": "Wes Anderson", "runtime": 99},
    "drama": {"title": "The Shawshank Redemption", "director": "Frank Darabont", "runtime": 142},
    "horror": {"title": "Get Out", "director": "Jordan Peele", "runtime": 104},
    "sci-fi": {"title": "Inception", "director": "Christopher Nolan", "runtime": 148}
}


def get_movie_recommendation(genre: str) -> str:
    """Get a movie recommendation for a given genre."""
    genre = genre.lower().strip()
    if genre in movie_database:
        movie = movie_database[genre]
        return f"'{movie['title']}' directed by {movie['director']} ({movie['runtime']} minutes)"
    else:
        available = ", ".join(movie_database.keys())
        return f"Sorry, no recommendations for '{genre}'. Available genres: {available}"


def get_runtime_category(minutes: str) -> str:
    """Categorize a movie's runtime and suggest viewing context."""
    try:
        runtime = int(minutes)
        if runtime < 100:
            return f"{runtime} minutes is a short film - perfect for a quick movie night."
        elif runtime < 130:
            return f"{runtime} minutes is a standard length - ideal for an evening watch."
        else:
            return f"{runtime} minutes is a longer film - make sure you have snacks ready!"
    except ValueError:
        return "Error: Please provide a valid number of minutes."


# Map action names to functions
available_actions = {
    "get_movie_recommendation": get_movie_recommendation,
    "get_runtime_category": get_runtime_category
}

print("Tools loaded successfully!")
print(f"Available actions: {list(available_actions.keys())}")

Tools loaded successfully!
Available actions: ['get_movie_recommendation', 'get_runtime_category']


In [5]:
# System prompt for the movie advisor agent
system_prompt = """
You run in a loop of Thought, Action, PAUSE, Observation.
At the end of the loop you output an Answer.

Use Thought to describe your reasoning about the question.
Use Action to run one of the available actions - then return PAUSE.
Observation will be the result of running those actions.
When you have enough information to answer the question, output your Answer.

Your available actions are:

get_movie_recommendation:
e.g. get_movie_recommendation: comedy
Returns a movie recommendation for the given genre.

get_runtime_category:
e.g. get_runtime_category: 120
Returns information about whether a movie runtime is short, standard, or long.

Example session:

Question: What's a good drama movie and is it a long watch?
Thought: I need to find a drama movie recommendation first, then check its runtime.
Action: get_movie_recommendation: drama
PAUSE

Observation: 'The Shawshank Redemption' directed by Frank Darabont (142 minutes)

Thought: Now I should check if 142 minutes is considered a long runtime.
Action: get_runtime_category: 142
PAUSE

Observation: 142 minutes is a longer film - make sure you have snacks ready!

Answer: I recommend 'The Shawshank Redemption' directed by Frank Darabont. At 142 minutes, it's a longer film, so make sure you have snacks ready!
""".strip()

print("System prompt loaded!")

System prompt loaded!


## Provided: Action Parser

This regex pattern extracts actions from the agent's response.

In [6]:
# Regex to parse actions from agent response
# Matches lines like: "Action: get_movie_recommendation: comedy"
action_re = re.compile(r'^Action: (\w+): (.*)$')

print("Action parser ready!")

Action parser ready!


## Your Task: Implement the Query Loop

Complete the `query` function below. This function should:

1. Create a new agent with the system prompt
2. Send the question to the agent
3. **Loop** until the agent stops returning actions:
   - Parse the response for actions using `action_re`
   - If an action is found, execute it and feed the observation back to the agent
   - If no action is found, the agent has finished (break the loop)
4. Use `max_turns` to prevent infinite loops

**Hints:**
- Use `action_re.match(line)` to check if a line contains an action
- Use `.groups()` on a match to get `(action_name, action_input)`
- Feed observations back as: `f"Observation: {observation}"`
- Print the agent's response and observations for visibility

In [7]:
def query(question, max_turns=5):
    """
    Run the ReAct loop until the agent outputs an Answer.

    Args:
        question: The user's question to answer
        max_turns: Maximum number of loop iterations (default 5)

    The function should:
    1. Create a MovieAdvisorAgent with system_prompt
    2. Send the question to the agent and print the response
    3. Loop up to max_turns times:
       - Parse response lines for actions using action_re
       - If action found: execute it, print observation, send observation to agent
       - If no action found: break (agent is done)
    """
    # TODO: Implement the query loop
    bot = MovieAdvisorAgent(system_prompt)
    next_prompt = question
    while max_turns > 0:
        max_turns -= 1
        response = bot(next_prompt)
        print(response)
        next_actions = [
            action_re.match(next_response) for next_response in response.split("\n") if action_re.match(next_response)
        ]
        if next_actions:
            action, action_input = next_actions[0].groups()
            if action not in available_actions:
                raise Exception(f"Unknown action: {action}: {action_input}")
            print(f" -- running {action}({action_input})")
            observation = available_actions[action](action_input)
            print("Observation:", observation)
            next_prompt = f"Observation: {observation}"
        else:
            break
    return response

## Test Your Implementation

Run the tests below to verify your loop implementation works correctly.

In [8]:
# Test 1: Single tool query
print("Test 1: Single tool query")
print("=" * 60)
query("What's a good comedy movie?")
print("=" * 60)

Test 1: Single tool query
Thought: I need to find a comedy movie recommendation to answer the question. 
Action: get_movie_recommendation: comedy
PAUSE
 -- running get_movie_recommendation(comedy)
Observation: 'The Grand Budapest Hotel' directed by Wes Anderson (99 minutes)
Thought: I have a comedy movie recommendation, which is 'The Grand Budapest Hotel' directed by Wes Anderson. Now, I should check its runtime to provide more information about it. 
Action: get_runtime_category: 99
PAUSE
 -- running get_runtime_category(99)
Observation: 99 minutes is a short film - perfect for a quick movie night.
Answer: I recommend 'The Grand Budapest Hotel' directed by Wes Anderson. At 99 minutes, it's a short film, perfect for a quick movie night!


In [9]:
# Test 2: Multi-tool query (requires loop!)
print("Test 2: Multi-tool query (this requires the loop to work!)")
print("=" * 60)
query("What's a good sci-fi movie and is it a long watch?")
print("=" * 60)

Test 2: Multi-tool query (this requires the loop to work!)
Thought: I need to find a sci-fi movie recommendation first, then check its runtime to determine if it's a long watch.  
Action: get_movie_recommendation: sci-fi  
PAUSE  
 -- running get_movie_recommendation(sci-fi  )
Observation: 'Inception' directed by Christopher Nolan (148 minutes)
Thought: Now I should check if 148 minutes is considered a long runtime.  
Action: get_runtime_category: 148  
PAUSE  
 -- running get_runtime_category(148  )
Observation: 148 minutes is a longer film - make sure you have snacks ready!
Answer: I recommend 'Inception' directed by Christopher Nolan. At 148 minutes, it's a longer film, so make sure you have snacks ready!


In [10]:
# Test 3: Another multi-tool query
print("Test 3: Another multi-tool query")
print("=" * 60)
query("I want a horror movie - will it be a quick watch?")
print("=" * 60)

Test 3: Another multi-tool query
Thought: I need to find a horror movie recommendation first, then check its runtime to determine if it's a quick watch.  
Action: get_movie_recommendation: horror  
PAUSE  
 -- running get_movie_recommendation(horror  )
Observation: 'Get Out' directed by Jordan Peele (104 minutes)
Thought: Now I should check if 104 minutes is considered a quick runtime.  
Action: get_runtime_category: 104  
PAUSE  
 -- running get_runtime_category(104  )
Observation: 104 minutes is a standard length - ideal for an evening watch.
Answer: I recommend 'Get Out' directed by Jordan Peele. At 104 minutes, it's a standard length, making it ideal for an evening watch.
